# Semana 7: rodada única do script de análise congelado

Seção 10 de `prereg/preregistro.md`. Roda `analysis/analise_congelada.py` nas margens das 15 execuções (`Mestrado/resultados/semana6/margens`), uma única vez. O notebook não calcula nada além do que o script faz e não altera o script.

**O que acontece com o cegamento.** O script lê a chave dos códigos só depois de processar todas as execuções, e a usa para rotular as tabelas. A linha `key read: R1=..., R2=..., R3=...` do log mostra a chave: é o momento em que o cegamento acaba, como o pré-registro prevê. Depois dela, as tabelas já saem com R1, R2 e R3. As células 1 a 4 não calculam métricas e não leem a chave.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, confere RAM (mínimo de 11 GB) e disco |
| 2 | Clona o repositório e confere os SHA-256 dos quatro arquivos do script com a tabela de `analysis/README.md`. Se algum diferir, para |
| 3 | Copia as margens para o disco local e confere cada arquivo pelo `manifesto_sha256.csv` |
| 4 | Confere as entradas contra `splits/` (contagens de unidades, janelas e pixels), sem calcular métrica |
| 5 | Roda o script (cerca de 4 horas), com `--sem-tta`, porque o TTA não foi feito |
| 6 | Copia a saída e o log para `resultados/semana6/analise` |

Abra o Colab logada na conta do mestrado e use um ambiente com pelo menos 11 GB de RAM. **Deixe a aba aberta durante a célula 5.** Se a sessão cair no meio, a rodada se perde; rode de novo desde a célula 1 (a célula 3 pula o que já foi copiado). **Se alguma célula der erro, pare, não mude nada no script e guarde a saída inteira.**

Depois da célula 6, me mande os arquivos de `resultados/semana6/analise`: `tabelas/principal.csv`, `sensibilidades.csv`, `eventos.csv`, `regimes.csv`, `precondicoes.csv`, `execucoes.csv`, `rq3_por_unidade.csv`, `resultados.json` e `analise.log`.

In [ ]:
import os, re, json, time, hashlib, shutil, subprocess, sys, csv
import psutil

from google.colab import drive
drive.mount('/content/drive')
RAIZ = '/content/drive/MyDrive/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS_DRIVE = f'{RAIZ}/resultados/semana6/margens'
CHAVE = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana6/analise'
assert os.path.isdir(MARGENS_DRIVE), 'Pasta resultados/semana6/margens não encontrada. Rode antes o notebook da semana 6.'
assert os.path.isfile(CHAVE), 'A chave dos códigos não existe. O arquivo só é conferido, não é lido aqui.'

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

ram = psutil.virtual_memory().total / 1e9
disco = shutil.disk_usage('/content')[2] / 1e9
print(f'RAM da sessão: {ram:.1f} GB (a análise usa cerca de 8 GB no pico). Disco local livre: {disco:.0f} GB.')
assert ram >= 11, 'RAM abaixo de 11 GB: a análise pode não caber. Use um ambiente com mais memória.'
assert disco > 30, 'Disco local com menos de 30 GB livres.'

In [ ]:
REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

# os quatro arquivos do script têm de ser os congelados em analysis/README.md
tabela = open(f'{REPO}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode a análise.'
print(f'Repositório no commit {commit[:7]}. Os quatro arquivos do script conferem com os SHA-256 de analysis/README.md.')

In [ ]:
LOCAL = '/content/margens'
os.makedirs(LOCAL, exist_ok=True)
manifesto = list(csv.DictReader(open(f'{MARGENS_DRIVE}/manifesto_sha256.csv', newline='')))
assert len(manifesto) == 64, f'O manifesto tem {len(manifesto)} arquivos, esperado 64 (60 execuções e 4 rótulos).'
t0 = time.time()
for linha in manifesto:
    destino = f'{LOCAL}/{linha["arquivo"]}'
    if not (os.path.exists(destino) and os.path.getsize(destino) == int(linha['bytes'])):
        shutil.copy(f'{MARGENS_DRIVE}/{linha["arquivo"]}', destino + '.parcial')
        os.replace(destino + '.parcial', destino)
    assert sha256_arquivo(destino) == linha['sha256'], f'{linha["arquivo"]} diferente do manifesto.'
extras = sorted(set(os.listdir(MARGENS_DRIVE)) - {l['arquivo'] for l in manifesto})
print(f'{len(manifesto)} arquivos copiados e conferidos pelo manifesto em {(time.time() - t0) / 60:.1f} min.')
print('Outros arquivos na pasta de margens (ignorados pela análise):', extras)

In [ ]:
# Conferência das entradas, sem calcular nenhuma métrica
sys.path.insert(0, f'{REPO}/analysis')
import analise_congelada as A
ctx = A.montar_contexto(REPO, LOCAL)
corridas = A.listar_corridas(LOCAL)
codigos = A.verificar_corridas(corridas)
print(f'Códigos: {" ".join(sorted(codigos))}; sementes por código: {[len(corridas[c]) for c in sorted(codigos)]}')
print(f'Unidades: fonte {ctx["fonte"].n}, alvo {ctx["alvo"].n} (Bolívia 15, T2 {ctx["alvo"].n - 15}).')
print('Entradas conferidas. Nenhuma métrica foi calculada.')
del ctx
import gc; gc.collect()

In [ ]:
# A rodada única do script congelado (cerca de 4 horas). Deixe esta aba aberta e o computador ligado.
# A chave dos códigos é lida pelo script só depois de processar todas as execuções, para rotular as tabelas.
SAIDA = '/content/analise'
LOG = '/content/analise.log'
cmd = (f'cd {REPO} && set -o pipefail && python analysis/analise_congelada.py --repo . --margens {LOCAL} '
       f'--chave "{CHAVE}" --saida {SAIDA} --sem-tta 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")

In [ ]:
assert os.path.isfile(f'{SAIDA}/resultados.json'), 'A análise não terminou: não há resultados.json. Guarde o log e me mande.'
os.makedirs(SAIDA_DRIVE, exist_ok=True)
shutil.copytree(SAIDA, SAIDA_DRIVE, dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/analise.log')
INFO = {'commit_repositorio': commit, 'scripts_sha256': registrados, 'comando': 'analise_congelada.py --sem-tta',
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2)
for raiz, _, arquivos in os.walk(SAIDA_DRIVE):
    for a in sorted(arquivos):
        print(os.path.relpath(os.path.join(raiz, a), SAIDA_DRIVE))
print(f'\nSaída copiada para {SAIDA_DRIVE}.')